# Módulo 10 · Aula 04 — Extração e Web Scraping

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O fornecedor manda a tabela de preços por e-mail, num Excel com três abas, cabeçalho na linha 7 e o preço como texto com vírgula. Toda segunda alguém passa duas horas copiando isso à mão."*

E a pior:

> *"O script de importação rodou de novo por engano e agora tem pedido duplicado no banco."*

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | CSV do mundo real | Encoding, separador, decimal |
| 2 | Excel | 🔴 O formato que mente |
| 3 | XML e JSON aninhado | Achatar sem perder nada |
| 4 | Extração de banco | Sem derrubar o servidor |
| 5 | 🎯 **Ingestão incremental** | A marca d'água |
| 6 | 🔴 **Idempotência** | Rodar duas vezes não duplica |
| 7 | Scraping com BeautifulSoup | HTML estático |
| 8 | Selenium | E quando ele é necessário |
| 9 | 🔴 **Ética e legalidade** | `robots.txt`, LGPD, termos de uso |

> 🎯 **A seção 6 é a mais importante do módulo.** Um pipeline que duplica dados ao rodar duas vezes é um pipeline que ninguém pode operar com tranquilidade.

## ⚙️ Preparação

O scraping desta aula acontece contra **páginas HTML criadas aqui mesmo** — nada de bater em site de terceiro para aprender.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

## 1. CSV do mundo real

In [ ]:
BASE = preparar("aula_10_04")
ENTRADA = BASE / "entrada"
ENTRADA.mkdir()

# O CSV que o ERP da Aurora exporta — como ele é de verdade
CSV_BR = ENTRADA / "vendas_erp.csv"
LINHAS_ERP = (
    "sku;descricao;quantidade;preco_unitario;data_venda\n"
    "NB-1000;Notebook Dell Inspiron;3;2.599,90;05/01/2026\n"
    "MO-1053;Monitor LG 24 polegadas;1;1.199,00;05/01/2026\n"
    "PE-1002;Mouse sem fio ergonômico;12;89,90;06/01/2026\n"
    "AR-1039;SSD 1TB de alta velocidade;2;429,00;06/01/2026\n"
    "PE-1007;Teclado ABNT2 com retroiluminação;5;249,90;07/01/2026\n"
)
# 🔑 Gravado em latin-1, como um ERP antigo faria
CSV_BR.write_bytes(LINHAS_ERP.encode("latin-1"))

print("O que o Pandas faz sem ajuda nenhuma:\n")
try:
    ruim = pd.read_csv(CSV_BR)
    print(ruim.to_string(index=False))
    print(f"\n   colunas detectadas: {len(ruim.columns)}  🔴 (deveriam ser 5)")
except Exception as erro:
    print(f"   🔴 {type(erro).__name__}: {str(erro)[:70]}")

In [ ]:
print("Com os quatro ajustes que o CSV brasileiro exige:\n")
bom = pd.read_csv(
    CSV_BR,
    sep=";",                 # 1. o Excel em PT-BR usa ponto e vírgula
    encoding="latin-1",      # 2. sistemas antigos não usam UTF-8
    decimal=",",             # 3. 2.599,90 é dois mil e quinhentos
    thousands=".",           #    ↑ e o ponto é separador de milhar
    dayfirst=True,
    parse_dates=["data_venda"],
)
print(bom.to_string(index=False))
print(f"\n   dtypes:\n{bom.dtypes.to_string()}")

print("""
🔑 Sem `decimal=","`, o `2.599,90` viraria TEXTO — e a soma daria erro
   ou, pior, concatenaria. Sem `dayfirst=True`, `05/01/2026` viraria
   5 de janeiro nos EUA e 1 de maio no Brasil.

   💭 Repare que nenhum desses erros ESTOURA. Eles produzem um número
      errado silenciosamente.
""")

In [ ]:
# 🔴 A detecção de encoding
print("A linha com acento, lida de três formas:\n")
for codificacao in ["latin-1", "cp1252", "utf-8"]:
    try:
        texto = CSV_BR.read_text(encoding=codificacao)
        acentuada = [l for l in texto.splitlines() if "PE-1007" in l][0]
        marca = "✅" if "retroiluminação" in acentuada else "🔴 corrompido"
        print(f"   {codificacao:<10} {acentuada.split(';')[1]:<36} {marca}")
    except UnicodeDecodeError as erro:
        print(f"   {codificacao:<10} {'—':<36} 🔴 UnicodeDecodeError "
              f"(byte {erro.object[erro.start]:#04x})")

print("""
⚠️ `latin-1` NUNCA falha — ele mapeia qualquer byte para algum
   caractere. Isso o torna útil como último recurso e perigoso como
   padrão: você lê sem erro e obtém texto corrompido.

💡 Para descobrir o encoding de verdade, use `charset-normalizer` ou
   `chardet`. E quando você controlar a origem, exija UTF-8.
""")

## 2. 🔴 Excel — o formato que mente

In [ ]:
# A planilha do fornecedor, com todos os vícios reais
XLSX = ENTRADA / "precos_fornecedor.xlsx"

with pd.ExcelWriter(XLSX, engine="openpyxl") as escritor:
    # Aba 1: cabeçalho na linha 7, com lixo em cima
    lixo = pd.DataFrame({
        "A": ["DISTRIBUIDORA XYZ LTDA", "Tabela de preços", "Vigência: 01/2026",
              "", "Contato: vendas@xyz.com.br", "", "codigo"],
        "B": ["", "", "", "", "", "", "descricao"],
        "C": ["", "", "", "", "", "", "preco"],
        "D": ["", "", "", "", "", "", "estoque"],
    })
    lixo.to_excel(escritor, sheet_name="Tabela", index=False, header=False)

    dados = pd.DataFrame({
        "A": ["NB-1000", "MO-1053", "PE-1002", "AR-1039", "", "Total:"],
        "B": ["Notebook Dell", "Monitor LG", "Mouse", "SSD 1TB", "", ""],
        "C": ["2599,90", "1199,00", "89,90", "429,00", "", "4317,80"],
        "D": [14, 31, 120, 47, None, 212],
    })
    dados.to_excel(escritor, sheet_name="Tabela", index=False,
                   header=False, startrow=7)

    pd.DataFrame({"obs": ["planilha auxiliar"]}).to_excel(
        escritor, sheet_name="Rascunho", index=False)

print("Abas do arquivo:", pd.ExcelFile(XLSX).sheet_names)
print("\nLeitura ingênua:\n")
print(pd.read_excel(XLSX).head(9).to_string())

In [ ]:
print("Leitura correta:\n")
precos = pd.read_excel(
    XLSX,
    sheet_name="Tabela",     # 1. a aba certa, pelo NOME
    skiprows=6,              # 2. pula o cabeçalho decorativo
    dtype={"codigo": str},   # 3. 🔴 código NUNCA é número
)
print(precos.to_string(index=False))

# 4. 🔴 remove a linha de TOTAL — ela não é um produto
precos = precos[precos["codigo"].notna() & (precos["codigo"] != "Total:")].copy()

# 5. o preço veio como texto com vírgula
precos["preco"] = (precos["preco"].astype(str)
                   .str.replace(".", "", regex=False)
                   .str.replace(",", ".", regex=False)
                   .astype(float))
precos["estoque"] = precos["estoque"].astype("int32")

print("\nDepois da limpeza:\n")
print(precos.to_string(index=False))
print(f"\n{precos.dtypes.to_string()}")

> 🔴 **A linha de "Total" é a armadilha mais cara do Excel.**
>
> Se ela entra no `DataFrame`, o seu faturamento **dobra** — porque você soma os itens e mais uma linha que já é a soma deles. E nada estoura: o número simplesmente fica errado.
>
> ⚠️ **E `dtype={"codigo": str}` não é preciosismo.** Um código como `00123` vira `123`, e o `JOIN` com o cadastro deixa de casar. Um SKU como `1E5` vira `100000.0` em notação científica — isso acontece de verdade, e o Excel faz isso sozinho ao abrir o arquivo.
>
> 🧭 **A regra: identificador é sempre texto.** CEP, CPF, SKU, código de barras, número de nota. Nenhum deles é usado em conta.

In [ ]:
# Conferência automática: a planilha mudou de formato?
def conferir_planilha(caminho: Path, colunas_esperadas: list[str],
                      aba: str, pular: int) -> pd.DataFrame:
    """Lê e FALHA CEDO se o formato mudou.

    💭 O fornecedor vai mudar o layout sem avisar. A pergunta não é
       "se", é "quando" — e você quer descobrir na importação, não no
       relatório do mês seguinte.
    """
    abas = pd.ExcelFile(caminho).sheet_names
    if aba not in abas:
        raise ValueError(f"🔴 aba '{aba}' não existe. Abas: {abas}")

    df = pd.read_excel(caminho, sheet_name=aba, skiprows=pular)
    faltando = set(colunas_esperadas) - set(df.columns)
    if faltando:
        raise ValueError(
            f"🔴 colunas ausentes: {sorted(faltando)}. Veio: {list(df.columns)}")
    return df


print("✅ formato esperado:")
conferir_planilha(XLSX, ["codigo", "descricao", "preco", "estoque"], "Tabela", 6)
print("   passou\n")

print("🔴 se o fornecedor renomear uma coluna:")
try:
    conferir_planilha(XLSX, ["codigo", "descricao", "preco_unitario"], "Tabela", 6)
except ValueError as erro:
    print(f"   {erro}")

## 3. XML e JSON aninhado

In [ ]:
# XML — o formato dos sistemas fiscais brasileiros
XML = ENTRADA / "pedidos.xml"
XML.write_text("""<?xml version="1.0" encoding="UTF-8"?>
<pedidos>
  <pedido id="9001" canal="site">
    <cliente><nome>Ana Prado</nome><cidade>Campinas</cidade></cliente>
    <data>2026-01-05</data>
    <itens>
      <item sku="NB-1000" quantidade="2" preco="2599.90"/>
      <item sku="PE-1002" quantidade="1" preco="89.90"/>
    </itens>
  </pedido>
  <pedido id="9002" canal="app">
    <cliente><nome>Bruno Lima</nome><cidade>São Paulo</cidade></cliente>
    <data>2026-01-06</data>
    <itens>
      <item sku="MO-1053" quantidade="1" preco="1199.00"/>
    </itens>
  </pedido>
</pedidos>
""", encoding="utf-8")

print("`pd.read_xml` resolve o caso simples:\n")
print(pd.read_xml(XML).to_string(index=False))
print("\n🔴 Mas repare: os ITENS sumiram. Ele achatou só o primeiro nível.")

In [ ]:
import xml.etree.ElementTree as ET

def achatar_pedidos(caminho: Path) -> pd.DataFrame:
    """Uma linha por ITEM, com os dados do pedido repetidos.

    🔑 Esse é o formato que a análise quer: cada linha é um evento
       completo. Repetir o cliente em cada item é desnormalização
       proposital (aula 10_01).
    """
    raiz = ET.parse(caminho).getroot()
    linhas = []
    for pedido in raiz.findall("pedido"):
        cliente = pedido.find("cliente")
        comum = {
            "pedido_id": int(pedido.get("id")),
            "canal": pedido.get("canal"),
            "data": pedido.findtext("data"),
            "cliente_nome": cliente.findtext("nome"),
            "cliente_cidade": cliente.findtext("cidade"),
        }
        for item in pedido.find("itens").findall("item"):
            linhas.append({**comum,
                           "sku": item.get("sku"),
                           "quantidade": int(item.get("quantidade")),
                           "preco": float(item.get("preco"))})
    return pd.DataFrame(linhas)


itens = achatar_pedidos(XML)
print("Achatado item a item:\n")
print(itens.to_string(index=False))
print(f"\n   2 pedidos → {len(itens)} linhas de item")

In [ ]:
# JSON aninhado — o que toda API devolve
resposta_api = {
    "pagina": 1,
    "pedidos": [
        {"id": 9001, "data": "2026-01-05",
         "cliente": {"nome": "Ana Prado", "endereco": {"cidade": "Campinas", "uf": "SP"}},
         "itens": [{"sku": "NB-1000", "q": 2, "preco": 2599.90},
                   {"sku": "PE-1002", "q": 1, "preco": 89.90}]},
        {"id": 9002, "data": "2026-01-06",
         "cliente": {"nome": "Bruno Lima", "endereco": {"cidade": "São Paulo", "uf": "SP"}},
         "itens": [{"sku": "MO-1053", "q": 1, "preco": 1199.00}]},
    ],
}

achatado = pd.json_normalize(
    resposta_api["pedidos"],
    record_path="itens",                 # 🔑 a lista que vira LINHAS
    meta=["id", "data",
          ["cliente", "nome"],
          ["cliente", "endereco", "cidade"]],   # 🔑 o que se REPETE
)
print("`json_normalize` faz o mesmo, declarativamente:\n")
print(achatado.to_string(index=False))

print("""
🔑 `record_path` é a lista que vira linhas.
   `meta` são os campos dos níveis de cima que se repetem em cada linha.

💡 Sem `meta`, você perde o vínculo: fica com itens sem saber de que
   pedido são.
""")

## 4. Extração de banco — sem derrubar o servidor

In [ ]:
# Montamos o banco de origem (o do M05)
ORIGEM = BASE / "origem.db"
conexao = sqlite3.connect(ORIGEM)
vendas = gerar_vendas(n=80_000, dias=200)
v = vendas.copy()
v["data"] = v["data"].dt.tz_localize(None)
# 🔑 Uma coluna de auditoria — quando a linha foi alterada pela última vez
v["atualizado_em"] = v["data"] + pd.to_timedelta(rng.integers(0, 3600, len(v)), unit="s")
v.to_sql("vendas", conexao, index=False, if_exists="replace")
conexao.execute("CREATE INDEX idx_atualizado ON vendas(atualizado_em)")
conexao.commit()

print(f"banco de origem: {len(v):,} linhas, {tamanho(ORIGEM.stat().st_size)}")
print(f"período: {v['atualizado_em'].min()} a {v['atualizado_em'].max()}")

In [ ]:
print("""
🔴 O QUE NÃO FAZER

   SELECT * FROM vendas

   · varre a tabela inteira toda vez
   · carrega tudo na memória do processo
   · concorre com o site (aula 10_01)
   · e ainda cresce todo dia

✅ TRÊS FORMAS MELHORES
""")

# 1. Só as colunas necessárias, em pedaços
def extrair_em_pedacos(tamanho_pedaco: int = 20_000):
    total = 0
    for pedaco in pd.read_sql(
            "SELECT pedido_id, data, sku, quantidade, preco_unitario, status "
            "FROM vendas", conexao, chunksize=tamanho_pedaco):
        total += len(pedaco)
    return total


linhas, ms = cronometrar(extrair_em_pedacos)
print(f"1. `chunksize` — memória constante")
print(f"   {linhas:,} linhas em {ms:.0f} ms\n")

# 2. Filtrar no BANCO, não no Python
def filtrar_no_python():
    d = pd.read_sql("SELECT * FROM vendas", conexao)
    return d[d["status"] == "pago"]

def filtrar_no_banco():
    return pd.read_sql("SELECT * FROM vendas WHERE status = 'pago'", conexao)

print("2. Filtrar no banco vs no Python:")
comparar([("filtro no Python", filtrar_no_python),
          ("filtro no BANCO", filtrar_no_banco)], rotulo="   onde filtrar")

## 5. 🎯 Ingestão incremental — a marca d'água

In [ ]:
print("""
   CARGA COMPLETA (full load)
   ══════════════════════════
   toda vez, tudo. Simples, e insustentável: o tempo cresce com a
   tabela, e um dia não cabe na janela da madrugada.

   CARGA INCREMENTAL
   ═════════════════
   só o que mudou DESDE a última vez.

   🔑 Precisa de duas coisas:
      1. uma coluna que diga QUANDO a linha mudou
      2. um lugar para guardar até onde você já leu  ← a MARCA D'ÁGUA
""")

MARCA = BASE / "marca_dagua.json"


def ler_marca(tabela: str) -> str | None:
    if not MARCA.exists():
        return None
    return json.loads(MARCA.read_text(encoding="utf-8")).get(tabela)


def gravar_marca(tabela: str, valor: str) -> None:
    atual = json.loads(MARCA.read_text(encoding="utf-8")) if MARCA.exists() else {}
    atual[tabela] = valor
    MARCA.write_text(json.dumps(atual, indent=2), encoding="utf-8")


def extrair_incremental(tabela: str = "vendas") -> pd.DataFrame:
    """Lê só o que mudou desde a última execução.

    ⚠️ O `>=` (e não `>`) é proposital: se duas linhas tiverem o mesmo
       instante e a execução anterior parou no meio, o `>` perderia a
       segunda. O `>=` relê algumas — e por isso a carga precisa ser
       IDEMPOTENTE (próxima seção).

       🎯 Preferir reler a perder é uma decisão consciente: duplicata
          você resolve com chave; dado perdido você não descobre.
    """
    desde = ler_marca(tabela)
    if desde is None:
        print("   primeira execução → carga completa")
        consulta = "SELECT * FROM vendas ORDER BY atualizado_em"
        parametros = ()
    else:
        print(f"   carga incremental desde {desde}")
        consulta = ("SELECT * FROM vendas WHERE atualizado_em >= ? "
                    "ORDER BY atualizado_em")
        parametros = (desde,)

    df = pd.read_sql(consulta, conexao, params=parametros,
                     parse_dates=["data", "atualizado_em"])
    if not df.empty:
        gravar_marca(tabela, df["atualizado_em"].max().isoformat())
    return df


print("── execução 1 ──")
lote1 = extrair_incremental()
print(f"   {len(lote1):,} linhas · marca: {ler_marca('vendas')}\n")

print("── execução 2 (nada mudou) ──")
lote2 = extrair_incremental()
print(f"   {len(lote2):,} linhas  ← só as do limite, por causa do >=")

In [ ]:
# Chegam vendas novas na origem
novas = gerar_vendas(n=500, dias=1)
novas["pedido_id"] = 900_000 + np.arange(len(novas))
n = novas.copy()
n["data"] = n["data"].dt.tz_localize(None)
n["atualizado_em"] = pd.Timestamp("2026-08-14 10:00:00") + pd.to_timedelta(
    np.arange(len(n)), unit="s")
n.to_sql("vendas", conexao, index=False, if_exists="append")

print("── 500 vendas novas chegaram ──")
print("── execução 3 ──")
lote3 = extrair_incremental()
print(f"   {len(lote3):,} linhas  ✅ só as novas")
print(f"   marca: {ler_marca('vendas')}")

print("""
🎯 A EXECUÇÃO 3 LEU 500 LINHAS EM VEZ DE 80.500.

   E o tempo passa a depender do VOLUME DIÁRIO, não do histórico —
   que é a única forma de a carga continuar cabendo na madrugada
   daqui a três anos.
""")

> ⚠️ **A marca d'água tem três armadilhas que só aparecem em produção:**
>
> | Armadilha | O que acontece | Defesa |
> |-----------|----------------|--------|
> | Linha inserida com data retroativa | Fica para trás da marca e **nunca é lida** | Reprocessar uma janela de segurança |
> | Relógios diferentes entre origem e destino | Perde ou repete registros | Use o relógio **da origem** |
> | Exclusão física na origem | A linha some e você não fica sabendo | *Soft delete* ou reconciliação periódica |
>
> 🔑 **A terceira é a mais séria.** Carga incremental **não detecta exclusão** — se o pedido 9042 for apagado na origem, ele continua no seu destino para sempre.
>
> 💭 A saída usual é uma reconciliação semanal: compare as contagens e as chaves, e trate as diferenças.

## 6. 🔴 Idempotência — rodar duas vezes não duplica

In [ ]:
DESTINO = BASE / "destino.db"
destino = sqlite3.connect(DESTINO)
destino.execute("""
    CREATE TABLE vendas (
        pedido_id      INTEGER PRIMARY KEY,   -- 🔑 a chave natural
        data           TEXT,
        sku            TEXT,
        quantidade     INTEGER,
        preco_unitario REAL,
        status         TEXT,
        atualizado_em  TEXT
    )
""")
destino.commit()

COLUNAS = ["pedido_id", "data", "sku", "quantidade",
           "preco_unitario", "status", "atualizado_em"]


def carregar_ingenuo(df: pd.DataFrame) -> int:
    """🔴 O jeito que duplica."""
    df[COLUNAS].astype({"data": str, "atualizado_em": str}).to_sql(
        "vendas_ingenuo", destino, index=False, if_exists="append")
    return len(df)


amostra = lote1.head(1000)
carregar_ingenuo(amostra)
n1 = destino.execute("SELECT COUNT(*) FROM vendas_ingenuo").fetchone()[0]
carregar_ingenuo(amostra)                       # 🔴 rodou de novo
n2 = destino.execute("SELECT COUNT(*) FROM vendas_ingenuo").fetchone()[0]

print("🔴 CARGA INGÊNUA — `to_sql(if_exists='append')`\n")
print(f"   depois da 1ª execução: {n1:,} linhas")
print(f"   depois da 2ª execução: {n2:,} linhas   🔴 DOBROU")
print(f"\n   E o faturamento do relatório dobra junto — sem nenhum erro.")

In [ ]:
def carregar_idempotente(df: pd.DataFrame) -> dict:
    """✅ UPSERT: insere o que é novo, atualiza o que mudou.

    🔑 `ON CONFLICT(chave) DO UPDATE` é o mesmo padrão do M03. Ele
       torna a carga IDEMPOTENTE: rodar N vezes dá o mesmo resultado
       que rodar uma.

    ⚠️ E ele exige que a tabela tenha PRIMARY KEY ou UNIQUE na chave
       natural — sem isso não há "conflito" a detectar.
    """
    registros = (df[COLUNAS]
                 .astype({"data": str, "atualizado_em": str})
                 .to_records(index=False).tolist())

    antes = destino.execute("SELECT COUNT(*) FROM vendas").fetchone()[0]
    destino.executemany(f"""
        INSERT INTO vendas ({', '.join(COLUNAS)})
        VALUES ({', '.join('?' * len(COLUNAS))})
        ON CONFLICT(pedido_id) DO UPDATE SET
            data           = excluded.data,
            sku            = excluded.sku,
            quantidade     = excluded.quantidade,
            preco_unitario = excluded.preco_unitario,
            status         = excluded.status,
            atualizado_em  = excluded.atualizado_em
        WHERE excluded.atualizado_em > vendas.atualizado_em
    """, registros)
    destino.commit()
    depois = destino.execute("SELECT COUNT(*) FROM vendas").fetchone()[0]
    return {"recebidas": len(df), "inseridas": depois - antes,
            "total": depois}


print("✅ CARGA IDEMPOTENTE — UPSERT\n")
for tentativa in (1, 2, 3):
    r = carregar_idempotente(amostra)
    print(f"   execução {tentativa}: recebeu {r['recebidas']:,} · "
          f"inseriu {r['inseridas']:,} · total {r['total']:,}")

print("\n   🎯 A tabela ficou igual nas três. Isso é idempotência.")

> 🔑 **Repare no `WHERE excluded.atualizado_em > vendas.atualizado_em`.**
>
> Ele só atualiza se o registro que chegou for **mais recente** que o gravado. Sem essa linha, um reprocessamento de dados antigos sobrescreveria correções recentes — e o dado "voltaria no tempo".
>
> 💭 **É o mesmo raciocínio do `keep='last'` no `drop_duplicates` (aula 10_02):** quando há duas versões do mesmo registro, a mais nova vence — e você precisa de um campo que diga qual é a mais nova.
>
> 🧭 **A regra geral da idempotência:** toda carga precisa de uma **chave natural** e de uma **regra de desempate**. Sem as duas, "rodar de novo" é uma aposta.

In [ ]:
# Provando que a atualização funciona
antes = destino.execute(
    "SELECT status FROM vendas WHERE pedido_id = ?",
    (int(amostra.iloc[0]["pedido_id"]),)).fetchone()[0]

corrigido = amostra.head(1).copy()
corrigido["status"] = "cancelado"
corrigido["atualizado_em"] = corrigido["atualizado_em"] + pd.Timedelta(days=1)
carregar_idempotente(corrigido)

depois = destino.execute(
    "SELECT status FROM vendas WHERE pedido_id = ?",
    (int(amostra.iloc[0]["pedido_id"]),)).fetchone()[0]

print(f"O pedido foi cancelado na origem e reenviado:\n")
print(f"   antes : {antes}")
print(f"   depois: {depois}   ✅ atualizou, não duplicou")
print(f"   total : {destino.execute('SELECT COUNT(*) FROM vendas').fetchone()[0]:,}")

## 7. Web scraping com BeautifulSoup

In [ ]:
from bs4 import BeautifulSoup

# Uma página como as de verdade — criada aqui, sem incomodar ninguém
PAGINA = ENTRADA / "catalogo.html"
PAGINA.write_text("""<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8"><title>Catálogo</title></head>
<body>
  <div class="lista-produtos">
    <article class="produto" data-sku="NB-1000">
      <h2 class="nome">Notebook Dell Inspiron 15</h2>
      <span class="preco">R$ 2.599,90</span>
      <span class="estoque em-estoque">14 unidades</span>
      <a href="/produto/NB-1000">detalhes</a>
    </article>
    <article class="produto" data-sku="MO-1053">
      <h2 class="nome">Monitor LG 24"</h2>
      <span class="preco">R$ 1.199,00</span>
      <span class="estoque esgotado">indisponível</span>
      <a href="/produto/MO-1053">detalhes</a>
    </article>
    <article class="produto" data-sku="PE-1002">
      <h2 class="nome">Mouse sem fio</h2>
      <span class="preco promocional">R$ 79,90</span>
      <span class="preco-antigo">R$ 89,90</span>
      <span class="estoque em-estoque">120 unidades</span>
      <a href="/produto/PE-1002">detalhes</a>
    </article>
  </div>
  <nav class="paginacao"><a href="?p=2" rel="next">próxima</a></nav>
</body></html>
""", encoding="utf-8")

sopa = BeautifulSoup(PAGINA.read_text(encoding="utf-8"), "html.parser")
print(f"produtos encontrados: {len(sopa.select('article.produto'))}")

In [ ]:
import re as _re


def dinheiro_br(texto: str) -> float | None:
    """'R$ 2.599,90' → 2599.90"""
    if not texto:
        return None
    limpo = _re.sub(r"[^\d,.-]", "", texto).replace(".", "").replace(",", ".")
    try:
        return float(limpo)
    except ValueError:
        return None


def extrair_catalogo(html: str) -> pd.DataFrame:
    """Extrai os produtos — com tolerância a campo ausente.

    🔑 Todo `select_one` pode devolver None. Numa página real, algum
       produto sempre vem sem preço, sem estoque ou sem link — e um
       `.text` em None derruba o raspador inteiro por causa de UM item.
    """
    sopa = BeautifulSoup(html, "html.parser")
    linhas = []
    for cartao in sopa.select("article.produto"):
        nome = cartao.select_one(".nome")
        preco = cartao.select_one(".preco")
        estoque = cartao.select_one(".estoque")
        linhas.append({
            "sku": cartao.get("data-sku"),
            "nome": nome.get_text(strip=True) if nome else None,
            "preco": dinheiro_br(preco.get_text() if preco else ""),
            "promocional": "promocional" in (preco.get("class", []) if preco else []),
            "disponivel": ("em-estoque" in estoque.get("class", [])) if estoque else None,
            "estoque_texto": estoque.get_text(strip=True) if estoque else None,
        })
    return pd.DataFrame(linhas)


catalogo = extrair_catalogo(PAGINA.read_text(encoding="utf-8"))
print(catalogo.to_string(index=False))

In [ ]:
print("""
💡 SELETORES — os que você vai usar

   sopa.select("article.produto")        todos (lista de CSS)
   sopa.select_one("h2.nome")            o primeiro (ou None)
   sopa.find_all("a", href=True)         por tag e atributo
   elemento.get("data-sku")              atributo (None se ausente)
   elemento.get_text(strip=True)         texto limpo

🔑 PREFIRA ATRIBUTO DE DADOS A POSIÇÃO.

   `article:nth-child(3) > span:nth-child(2)` quebra na primeira
   mudança de layout. `[data-sku]` sobrevive a redesenho.

⚠️ E prefira `html.parser` (embutido) ou `lxml` (rápido) —
   `html5lib` é o mais tolerante e o mais lento.
""")

# Paginação
proxima = sopa.select_one('nav.paginacao a[rel="next"]')
print(f"   próxima página: {proxima.get('href') if proxima else 'não há'}")
print("""
🔑 O padrão de paginação é o gerador do M07: siga o link "próxima"
   até ele não existir mais — e ponha um TETO de páginas.
""")

In [ ]:
# `read_html` para o caso fácil
TABELA = ENTRADA / "tabela.html"
TABELA.write_text("""<table>
  <tr><th>sku</th><th>categoria</th><th>vendas</th></tr>
  <tr><td>NB-1000</td><td>Notebooks</td><td>142</td></tr>
  <tr><td>MO-1053</td><td>Monitores</td><td>89</td></tr>
</table>""", encoding="utf-8")

import io
tabelas = pd.read_html(io.StringIO(TABELA.read_text(encoding="utf-8")))
print(f"`pd.read_html` achou {len(tabelas)} tabela(s):\n")
print(tabelas[0].to_string(index=False))
print("""
💡 Quando o dado JÁ está numa `<table>`, `read_html` resolve numa
   linha. Só não confie nos dtypes — passe `dtype=` como no CSV.
""")

## 8. Selenium — e quando ele é necessário

In [ ]:
print("""
   HTML ESTÁTICO                    RENDERIZADO POR JAVASCRIPT
   ═════════════                    ══════════════════════════
   o conteúdo vem no HTML           o HTML vem quase vazio
   requests + BeautifulSoup         Selenium / Playwright
   milissegundos                    🔶 segundos por página
   sem navegador                    🔶 um Chrome por processo

🔑 COMO SABER QUAL É?

   1. veja o código-fonte (Ctrl+U) — não o inspetor
   2. se o dado aparece lá  → estático, use requests
   3. se não aparece        → é JavaScript

⚠️ MAS ANTES DE PARTIR PARA O SELENIUM, PROCURE A API.

   Abra a aba Rede do navegador e recarregue. Quase sempre existe uma
   chamada a `/api/produtos?pagina=2` devolvendo JSON limpo — o mesmo
   que o JavaScript usa para montar a página.

   🎯 Consumir essa API é mais rápido, mais estável e mais educado
      que renderizar a página inteira. E você já sabe fazer isso (M07).
""")

print("""
📖 SELENIUM — REFERÊNCIA (não roda aqui: exige navegador)

   from selenium import webdriver
   from selenium.webdriver.common.by import By
   from selenium.webdriver.support.ui import WebDriverWait
   from selenium.webdriver.support import expected_conditions as EC

   opcoes = webdriver.ChromeOptions()
   opcoes.add_argument("--headless=new")
   navegador = webdriver.Chrome(options=opcoes)
   try:
       navegador.get("https://exemplo.com/catalogo")

       # 🔴 NUNCA time.sleep(5). Espere a CONDIÇÃO, não o relógio.
       WebDriverWait(navegador, 10).until(
           EC.presence_of_element_located((By.CSS_SELECTOR, "article.produto")))

       # 💡 Depois de renderizado, volte para o BeautifulSoup:
       sopa = BeautifulSoup(navegador.page_source, "html.parser")
   finally:
       navegador.quit()        # 🔴 senão o Chrome fica na memória

   ⚠️ `time.sleep` fixo é o erro nº 1: lento demais quando a página
      responde rápido, curto demais quando ela demora — e aí o
      raspador falha de forma intermitente.
""")

## 9. 🔴 Ética e legalidade

In [ ]:
print("""
🔴 ANTES DE RASPAR QUALQUER SITE, TRÊS PERGUNTAS:

   1. O `robots.txt` permite?
   2. Os termos de uso permitem?
   3. Existe uma API oficial?

   Se a resposta da 3 for sim, use a API. Fim da discussão.
""")

import urllib.robotparser

EXEMPLO_ROBOTS = ENTRADA / "robots.txt"
EXEMPLO_ROBOTS.write_text("""User-agent: *
Disallow: /admin/
Disallow: /checkout/
Crawl-delay: 2

User-agent: BotAgressivo
Disallow: /
""", encoding="utf-8")

leitor = urllib.robotparser.RobotFileParser()
leitor.parse(EXEMPLO_ROBOTS.read_text(encoding="utf-8").splitlines())

print("Consultando o robots.txt programaticamente:\n")
for agente, caminho in [("atlas-bot", "/catalogo"),
                        ("atlas-bot", "/admin/usuarios"),
                        ("BotAgressivo", "/catalogo")]:
    permitido = leitor.can_fetch(agente, caminho)
    print(f"   {agente:<14} {caminho:<18} {'✅ pode' if permitido else '🔴 não pode'}")

print(f"\n   Crawl-delay pedido: {leitor.crawl_delay('atlas-bot')} segundos")

In [ ]:
print("""
🧭 AS REGRAS DE UM RASPADOR EDUCADO

   ✅ identifique-se no User-Agent, com e-mail de contato
   ✅ respeite o robots.txt e o Crawl-delay
   ✅ 1 requisição por segundo, no máximo (o M07 já ensinou o limitador)
   ✅ guarde o HTML bruto — para não precisar raspar de novo
   ✅ use cache: não peça a mesma página duas vezes
   ✅ raspe fora do horário de pico do site

   🔴 NÃO burle CAPTCHA, login ou paywall
   🔴 NÃO raspe dado pessoal — LGPD vale aqui também
   🔴 NÃO republique o conteúdo como se fosse seu
   🔴 NÃO derrube o site alheio com volume

💭 E A PARTE INCÔMODA:

   "É público" NÃO significa "pode usar para qualquer coisa". Dado
   pessoal exposto numa página continua sendo dado pessoal, e a LGPD
   se aplica a quem COLETA — inclusive por raspagem.

   ⚠️ Se o seu raspador coleta nome, e-mail, telefone ou CPF de
      pessoas, você precisa de base legal para isso. Converse com
      alguém de jurídico antes, não depois.
""")

In [ ]:
# Um cabeçalho honesto — e o limitador do M07
CABECALHOS = {
    "User-Agent": "atlas-bot/1.0 (+https://aurora.com.br/bot; engenharia@aurora.com.br)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "pt-BR,pt;q=0.9",
}
for chave, valor in CABECALHOS.items():
    print(f"   {chave:<18}{valor}")

print("""
🔑 O `+https://...` no User-Agent é convenção: uma página explicando
   quem você é e como pedir para parar.

   💭 Isso não é só cortesia — é interesse próprio. Um administrador
      que consegue te contatar pede para você reduzir o ritmo. Um que
      não consegue simplesmente bloqueia o seu IP.
""")

## 🔧 Prática guiada — o extrator do fornecedor

In [ ]:
def extrair_precos_fornecedor(caminho: Path) -> tuple[pd.DataFrame, dict]:
    """Lê a planilha do fornecedor com todas as defesas.

    Devolve (dados_limpos, relatorio) — nunca levanta por dado ruim.
    Linha ruim vai para o relatório, não para o resultado.
    """
    relatorio = {"arquivo": caminho.name, "lidas": 0, "validas": 0,
                 "descartadas": [], "avisos": []}

    abas = pd.ExcelFile(caminho).sheet_names
    if "Tabela" not in abas:
        raise ValueError(f"🔴 aba 'Tabela' não encontrada. Abas: {abas}")

    df = pd.read_excel(caminho, sheet_name="Tabela", skiprows=6,
                       dtype={"codigo": str})
    relatorio["lidas"] = len(df)

    esperadas = {"codigo", "descricao", "preco", "estoque"}
    faltando = esperadas - set(df.columns)
    if faltando:
        raise ValueError(f"🔴 colunas ausentes: {sorted(faltando)}")

    # 🔴 Fora as linhas que não são produto
    nao_produto = df["codigo"].isna() | df["codigo"].astype(str).str.contains(
        r"total|soma|subtotal", case=False, na=False)
    for _, linha in df[nao_produto].iterrows():
        relatorio["descartadas"].append(
            {"motivo": "não é produto", "codigo": str(linha["codigo"])})
    df = df[~nao_produto].copy()

    # Preço em texto brasileiro
    df["preco"] = (df["preco"].astype(str)
                   .str.replace(r"[^\d,.-]", "", regex=True)
                   .str.replace(".", "", regex=False)
                   .str.replace(",", ".", regex=False))
    df["preco"] = pd.to_numeric(df["preco"], errors="coerce")

    sem_preco = df["preco"].isna()
    for _, linha in df[sem_preco].iterrows():
        relatorio["descartadas"].append(
            {"motivo": "preço ilegível", "codigo": linha["codigo"]})
    df = df[~sem_preco].copy()

    # Conferências de sanidade
    if (df["preco"] <= 0).any():
        relatorio["avisos"].append("há preços zerados ou negativos")
    if df["codigo"].duplicated().any():
        relatorio["avisos"].append("há códigos repetidos na planilha")

    df["estoque"] = pd.to_numeric(df["estoque"], errors="coerce").fillna(0).astype("int32")
    df["extraido_em"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

    relatorio["validas"] = len(df)
    return df.reset_index(drop=True), relatorio


dados, relatorio = extrair_precos_fornecedor(XLSX)
print(dados.to_string(index=False))
print(f"\nRelatório da extração:")
print(json.dumps(relatorio, indent=2, ensure_ascii=False))

> 🎯 **O relatório é tão importante quanto os dados.**
>
> Ele responde, sem ninguém precisar investigar: *quantas linhas vieram, quantas serviram, o que foi descartado e por quê.*
>
> 💭 **E repare que a função NÃO levanta exceção por linha ruim.** Uma linha com preço ilegível não pode derrubar a importação inteira — ela vai para o relatório, e o resto entra. Só o que impede a leitura (aba ausente, coluna faltando) é erro fatal.
>
> 🧭 **A distinção:** erro de **formato** para tudo; erro de **conteúdo** vira quarentena. Você vai formalizar isso na próxima aula.

In [ ]:
conexao.close()
destino.close()
print("Arquivos gerados:\n")
for caminho in sorted(BASE.rglob("*")):
    if caminho.is_file():
        print(f"   {str(caminho.relative_to(BASE)):<34}{tamanho(caminho.stat().st_size):>12}")

## 📝 Exercícios

**E1.** Leia um CSV brasileiro com os quatro ajustes. Mostre o que acontece sem cada um deles.

**E2.** 🔴 Leia o mesmo arquivo com três encodings diferentes. Explique por que `latin-1` nunca falha.

**E3.** Leia uma planilha com cabeçalho na linha 7 e linha de total. Mostre o faturamento errado se a linha de total entrar.

**E4.** 🔴 Mostre um código `00123` virando `123`. Corrija com `dtype`.

**E5.** Escreva um verificador que falhe se a planilha mudar de layout.

**E6.** Achate um XML de dois níveis para uma linha por item.

**E7.** Use `json_normalize` com `record_path` e `meta` numa resposta de API aninhada.

**E8.** Compare filtrar no banco e filtrar no Python. Explique a diferença.

**E9.** 🎯 Implemente uma carga incremental com marca d'água. Rode três vezes e mostre a terceira lendo pouco.

**E10.** 🔴 Explique as três armadilhas da marca d'água e proponha defesa para cada uma.

**E11.** 🔴 Demonstre a duplicação com `if_exists='append'`. Corrija com UPSERT.

**E12.** Explique o `WHERE excluded.atualizado_em > ...` do UPSERT. O que quebra sem ele?

**E13.** Extraia produtos de um HTML com BeautifulSoup, tolerando campos ausentes.

**E14.** Implemente a paginação seguindo o `rel="next"`, com teto de páginas.

**E15.** Consulte um `robots.txt` com `urllib.robotparser` e respeite o `Crawl-delay`.

**E16.** 🔴 Escreva a política de raspagem da Aurora: o que pode, o que não pode, e por quê.

**E17.** Escreva um extrator que devolva dados **e** relatório, sem levantar por linha ruim.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

In [ ]:
# E17

## 📋 Cola de referência

```python
# ═══ CSV brasileiro ═══
pd.read_csv(f, sep=";", encoding="latin-1", decimal=",", thousands=".",
            dayfirst=True, parse_dates=["data"], dtype={"codigo": str})
# ⚠️ latin-1 NUNCA falha → lê sem erro e corrompe em silêncio

# ═══ 🔴 Excel ═══
pd.read_excel(f, sheet_name="Tabela", skiprows=6, dtype={"codigo": str})
# 🔴 remova a linha de TOTAL — senão o faturamento dobra
# 🔴 identificador é SEMPRE str (CEP, CPF, SKU, nota)

# ═══ Aninhado ═══
pd.json_normalize(dados, record_path="itens",
                  meta=["id", ["cliente", "nome"]])
ET.parse(f).getroot()          # XML com mais de um nível

# ═══ Banco ═══
pd.read_sql(sql, con, chunksize=20_000)     # memória constante
# 🔑 filtre no BANCO, não no Python
# 🔑 só as colunas que usa

# ═══ 🎯 Incremental ═══
# 1. coluna de auditoria (atualizado_em)
# 2. marca d'água guardada
WHERE atualizado_em >= ?        # >= e não > : prefira reler a perder
# ⚠️ não detecta EXCLUSÃO → reconciliação periódica

# ═══ 🔴 Idempotência ═══
INSERT ... ON CONFLICT(chave) DO UPDATE SET ...
WHERE excluded.atualizado_em > tabela.atualizado_em
# 🔴 to_sql(if_exists="append") DUPLICA ao rodar duas vezes

# ═══ Scraping ═══
sopa.select("article.produto")      ·  sopa.select_one(".nome")
elemento.get("data-sku")            ·  .get_text(strip=True)
# 🔑 atributo de dados > posição · todo select_one pode ser None
# 🔴 antes do Selenium, procure a API na aba Rede

# ═══ 🔴 Ética ═══
urllib.robotparser.RobotFileParser()   # can_fetch, crawl_delay
# User-Agent com contato · 1 req/s · guarde o HTML bruto
# 🔴 "é público" ≠ "pode usar" — LGPD vale para quem COLETA
```

## ✅ Checklist de saída

**Arquivos**

- [ ] Leio CSV brasileiro com `sep`, `encoding`, `decimal`, `thousands`
- [ ] Sei por que `latin-1` é perigoso como padrão
- [ ] 🔴 **Removo a linha de total das planilhas**
- [ ] 🔴 **Leio identificador como texto**
- [ ] Falho cedo quando o layout muda
- [ ] Achato XML e JSON aninhado sem perder o vínculo

**Banco**

- [ ] Filtro no banco, não no Python
- [ ] Leio em pedaços
- [ ] 🎯 **Implemento carga incremental com marca d'água**
- [ ] Sei que ela não detecta exclusão

**Carga**

- [ ] 🔴 **Minha carga é idempotente (UPSERT)**
- [ ] Tenho chave natural e regra de desempate
- [ ] Rodar duas vezes dá o mesmo resultado

**Scraping**

- [ ] Uso seletores por atributo, não por posição
- [ ] Trato campo ausente sem derrubar tudo
- [ ] Sei distinguir HTML estático de renderizado
- [ ] Procuro a API antes de partir para o Selenium
- [ ] Nunca uso `sleep` fixo para esperar

**Ética**

- [ ] 🔴 **Consulto o `robots.txt`**
- [ ] Identifico-me no `User-Agent`, com contato
- [ ] Limito o ritmo
- [ ] 🔴 **Sei que "é público" não autoriza tudo, e que a LGPD se aplica**

---

### ➡️ Próxima aula

**`10_05_Arquitetura_ETL_e_Qualidade.ipynb`** — Juntar tudo num pipeline que roda sozinho de madrugada, sobrevive a dado ruim e pode ser reprocessado sem medo.